# Forecast next month

**The job.** Two years of monthly sales. Predict the next three months.

There is one way to get this wrong that beats all the others: split the data at
random. Every row after the split leaks into training, the score looks superb,
and the model is useless in production.

So the split is a step in the graph with two named outputs, and you can see in
the picture which one feeds the features.

In [ ]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

In [ ]:
import numpy as np

months = 24
t = np.arange(months)
trend = 120 + 4.5 * t
season = 25 * np.sin(2 * np.pi * t / 12)
noise = np.random.default_rng(4).normal(0, 6, months)
sales = (trend + season + noise).round(1)

print(f"{'month':>6}{'sales':>9}")
for i in list(range(3)) + [-2, -1]:
    print(f"{i % months:>6}{sales[i]:>9.1f}")
print(f"\n{months} months, {sales.min():.0f} to {sales.max():.0f}")

In [ ]:
nodes = [
    node("load.series",  "read",     [],                    [("out", "Series")]),
    node("split.time",   "split",    [("in", "Series")],    [("train", "Series"), ("test", "Series")]),
    node("split.random", "split",    [("in", "Series")],    [("train", "Series"), ("test", "Series")],
         runtime={"deterministic": False}),
    node("feat.trend",   "features", [("in", "Series")],    [("out", "Matrix")]),
    node("fit.linear",   "fit",      [("in", "Matrix")],    [("out", "Model")]),
    node("check.ahead",  "check",    [("in", "Model")],     [("out", "Score")]),
]

stages = [
    stage("load",  "Load the series", [],                 [("out", "Series")], "read",     ["load.series"]),
    stage("split", "Hold months back",[("in", "Series")], [("train", "Series"), ("test", "Series")], "split",
          ["split.time", "split.random"]),
    stage("feat",  "Build features",  [("in", "Series")], [("out", "Matrix")], "features", ["feat.trend"]),
    stage("fit",   "Fit",             [("in", "Matrix")], [("out", "Model")],  "fit",      ["fit.linear"]),
    stage("check", "Score ahead",     [("in", "Model")],  [("out", "Score")],  "check",    ["check.ahead"]),
]

edges = [Edge("load", "split"), Edge("split", "feat", from_port="train"),
         Edge("feat", "fit"), Edge("fit", "check")]

bench = build("Forecast next month",
              "Predict the next three months without letting the future leak in.",
              stages, nodes, edges)

In [ ]:
viz.dag(bench)

The arrow from `split` is labelled `train`. That is the whole safety property,
and it is visible in the picture rather than buried in a line of code.

In [ ]:
HOLD = 6

def load_series():
    return {"t": t, "y": sales}

def split_time(**kw):
    """By time. The last six months are the future and stay unseen."""
    s = kw["in"]
    return {"train": {"t": s["t"][:-HOLD], "y": s["y"][:-HOLD]},
            "test":  {"t": s["t"][-HOLD:], "y": s["y"][-HOLD:]}}

def split_random_bad(**kw):
    """The wrong one, kept so the difference can be measured rather than asserted."""
    s = kw["in"]
    order = np.random.default_rng(1).permutation(len(s["t"]))
    keep, hold = order[:-HOLD], order[-HOLD:]
    return {"train": {"t": s["t"][keep], "y": s["y"][keep]},
            "test":  {"t": s["t"][hold], "y": s["y"][hold]}}

def feat_trend(**kw):
    s = kw["in"]
    X = np.column_stack([np.ones(len(s["t"])), s["t"],
                         np.sin(2 * np.pi * s["t"] / 12),
                         np.cos(2 * np.pi * s["t"] / 12)])
    return {"X": X, "y": s["y"], "t": s["t"]}

def fit_linear(**kw):
    d = kw["in"]
    w, *_ = np.linalg.lstsq(d["X"], d["y"], rcond=None)
    return {"weights": w, "data": d}

def check_ahead(**kw):
    m = kw["in"]
    future = np.arange(months, months + 3)
    Xf = np.column_stack([np.ones(3), future,
                          np.sin(2 * np.pi * future / 12),
                          np.cos(2 * np.pi * future / 12)])
    fitted = m["data"]["X"] @ m["weights"]
    return {"in_sample_mae": float(np.abs(m["data"]["y"] - fitted).mean()),
            "forecast": (Xf @ m["weights"]).round(1).tolist(),
            "months": future.tolist()}

runtime = execute.Runtime({
    "load.series": load_series, "split.time": split_time,
    "split.random": split_random_bad, "feat.trend": feat_trend,
    "fit.linear": fit_linear, "check.ahead": check_ahead})

base = {s.id: s.candidates[0] for s in bench.leaf_stages}
proper = compile_route(bench, dict(base, split="split.time"))
leaky = compile_route(bench, dict(base, split="split.random"))

good = execute.run(proper, runtime)
bad = execute.run(leaky, runtime)

print(f"{'split':<16}{'deterministic':<15}{'in-sample error':>16}")
print(f"{'by time':<16}{str(proper.deterministic):<15}{good.output('check')['in_sample_mae']:>16.2f}")
print(f"{'at random':<16}{str(leaky.deterministic):<15}{bad.output('check')['in_sample_mae']:>16.2f}")

The random split is marked **not deterministic** by the plan, without anyone
saying so in this cell. It read that off the node.

## The forecast

In [ ]:
result = good.output("check")
print(f"{'month':>6}{'forecast':>11}")
for m, value in zip(result["months"], result["forecast"]):
    print(f"{m:>6}{value:>11.1f}")

held = split_time(**{"in": load_series()})["test"]
print(f"\nlast six actual: {[float(v) for v in held['y']]}")